# Diabetes Foot Risk Prediction Model — Google Colab walkthrough

This notebook sets up the published Fine-Gray competing-risk model, exports the official fitted R model once, and makes predictions with the Python package. Follow the cells in order. **The export cell requires R because the original validated model is an RDS file; once the JSON artifact exists, prediction runs in Python.**

> **Clinical safety:** This is a research/educational demonstration, not a medical device or diagnosis. Do not use its output to make individual care decisions. Do not upload identifiable or confidential patient data to a hosted Colab runtime; use only approved, de-identified data and your organization’s policies.


## Step 1 — Start a Colab runtime

Open this notebook in Google Colab, then select **Runtime → Run all** or execute each cell in order. A standard CPU runtime is sufficient. The runtime is temporary: generated files disappear when it disconnects, so download the JSON/model outputs you need.

In [ ]:
from pathlib import Path
import os

WORK = Path('/content')
APP_REPO = WORK / 'pirddiiqlabdecisions'
MODEL_REPO = WORK / 'PIRDDIIQ-FootCare-risk-model'

!git clone --depth 1 https://github.com/contrivancecompanychicago/pirddiiqlabdecisions.git {APP_REPO}
!git clone --depth 1 https://github.com/contrivancecompanychicago/PIRDDIIQ-FootCare-risk-model.git {MODEL_REPO}
print('Python package:', APP_REPO)
print('Original trained model:', MODEL_REPO / 'model/final_FGR_clean.rds')

## Step 2 — Install R and the one-time export dependencies

The trained coefficients, spline knots, and baseline curve are in the upstream `.rds` file. This cell installs R and the packages needed to convert that official model to a portable JSON file. It may take several minutes. You do not need to learn R or edit the R export script.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y r-base r-base-dev
!Rscript -e "install.packages(c('riskRegression', 'rms', 'jsonlite'), repos='https://cloud.r-project.org')"

## Step 3 — Export the original fitted model to JSON

The exporter checks that the original model's coefficient names and feature-matrix columns agree. If it reports a mismatch or another error, stop and investigate rather than using a guessed/approximate model. The JSON is created inside this Colab session, not committed to this package repository.

In [ ]:
ARTIFACT = APP_REPO / 'footcare_risk_model' / 'footcare_fgr_model.json'
export_script = APP_REPO / 'footcare_risk_model' / 'export_model.R'
!cd {MODEL_REPO} && Rscript {export_script} {ARTIFACT}
assert ARTIFACT.exists(), 'Export did not create the model JSON; read the R error above.'
print(f'Exported artifact: {ARTIFACT} ({ARTIFACT.stat().st_size:,} bytes)')

## Step 4 — Install and import the Python package

The package handles the validated model's restricted cubic spline features, lab missingness flags, and cumulative risk calculation.

In [ ]:
!python -m pip install -q -e {APP_REPO}
import pandas as pd
from footcare_risk_model import FootCareRiskModel
model = FootCareRiskModel(ARTIFACT)
print('Python model is ready; supported horizon:', model.times[-1], 'days')

## Step 5 — Try the published example patient

This is the hypothetical patient used in the upstream R documentation. At 365.25 days, the published reference is about **0.0171758** (1.72%). A different result is a reason to stop and check the exported artifact and implementation; do not treat it as clinical validation.

In [ ]:
example_patient = pd.DataFrame([
    {'age': 67, 'sex_f': 1, 'elective_adm': 1, 'homelessness': 0,
     'peripheral_AD': 0, 'coronary_AD': 1, 'stroke': 0, 'CHF': 0,
     'hypertension': 1, 'COPD': 0, 'CKD': 0, 'malignancy': 0,
     'mental_illness': 0, 'creatinine': 140.0, 'Hb_A1C': 8.5, 'albumin': 32.1}
])
example_risk = model.predict(example_patient, 365.25)[0]
print(f'Illustrative 1-year risk: {example_risk:.6f} ({example_risk:.2%})')
print('Upstream documentation reference: 0.0171758')

## Step 6 — Score a CSV file (optional)

Prepare a CSV with the predictor columns in the upstream data dictionary: `age`, `sex_f`, `elective_adm`, `homelessness`, `peripheral_AD`, `coronary_AD`, `stroke`, `CHF`, `hypertension`, `COPD`, `CKD`, `malignancy`, `mental_illness`, `creatinine`, `Hb_A1C`, and `albumin`. Binary columns must be 0/1 (or booleans); blank lab values are imputed to zero with a missingness flag. Ensure units and variable definitions match the data dictionary.

Use de-identified sample data only in a hosted notebook. Run the next cell to select a CSV from your computer, score at one year, and download the results.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose a CSV with the required predictor columns
input_name = next(iter(uploaded))
patients = pd.read_csv(input_name)
patients['foot_complication_risk_1_year'] = model.predict(patients, 365.25)
output_name = 'footcare_risk_predictions.csv'
patients.to_csv(output_name, index=False)
files.download(output_name)

## Step 7 — Run package tests (optional)

These tests use a synthetic fixture to check package behavior; they do not validate clinical performance.

In [ ]:
!cd {APP_REPO} && python -m pytest tests/test_footcare_risk_model.py -q

## Files and troubleshooting

- Python package and instructions: `footcare_risk_model/`
- One-time R exporter: `footcare_risk_model/export_model.R`
- Python package tests: `tests/test_footcare_risk_model.py`
- Original model, predictor definitions, and demo: [upstream repository](https://github.com/contrivancecompanychicago/PIRDDIIQ-FootCare-risk-model).

If R package installation or export fails, inspect the first R error and do not substitute arbitrary coefficients. For large production or sensitive datasets, do not use Colab; ask your institution's data/security team for an approved environment.